## **Claude SDK Tutorial**

https://anthropic.skilljar.com/claude-with-the-anthropic-api

---

In [1]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Make a request
def chat(messages: list[object], system_prompt=None, temperature=1.0, stop_sequences=[]) -> str:

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system_prompt:
        params["system"] = system_prompt

    message = client.messages.create(**params) # unpack dict into keyword args
    return message.content[0].text

In [9]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "What is quantum computing")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, final_answer)

# Print the full message history
# messages

# Print final answer
final_answer

'Quantum computing could eventually transform industries from healthcare to finance by solving optimization problems that are currently impossible for even the most powerful supercomputers to tackle.'

In [ ]:
# Implement looping chatbot
messages = []

while True:
    user_input = input("> ")
    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print(">", answer)
    print("---")

> 2 * 10 = 20
---
> 20 + 99 = 119
---


KeyboardInterrupt: Interrupted by user

In [11]:
messages = []

add_user_message(messages, "write a python function that checks string for duplicate characters")

answer = chat(messages, system_prompt="You are a python engineer who writes very concise code")

answer

'```python\ndef has_duplicates(s):\n    return len(s) != len(set(s))\n```\n\nThis function converts the string to a set (which removes duplicates) and compares the lengths. If they differ, duplicates exist.'

---

## **Understanding Temperature**

![temperature](temperature.png)

In [15]:
messages = []

add_user_message(messages, "tell a joke")

answer = chat(messages, temperature=1.0)

answer

"Why don't scientists trust atoms?\n\nBecause they make up everything!"

---

## **Streaming**

In [ ]:
# Manaul streaming
messages = []

add_user_message(messages, "Write one sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01Ahued3ArKfkdVikrnjwo7C', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=8, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text="Here's a one-sentence description of", type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=' a fake database:\n\n"', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='P', type='text_delta'), index=0, type='content_block

In [ ]:
# Built-in streaming capability
messages = []

add_user_message(messages, "Write one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")
        # pass

# Collect all events e.g., For record keeping in DB
# stream.get_final_message()

A fake database is a simulated or mock data storage system that contains artificially generated or placeholder information used for testing, development, or demonstration purposes without containing real user or production data.

---

## **Structured Data Using Prefill Assistant Message & Stop-Sequences**

In [ ]:
messages = []

add_user_message(messages, "Generate very short event bridge rule as json")

# Prefill assistant message
add_assistant_message(messages, "```json") # writes out response after the text containing "json"

# Stop-sequence
text = chat(messages, stop_sequences=["```"])

text

'\n{\n  "Name": "MyRule",\n  "EventPattern": {\n    "source": ["myapp"],\n    "detail-type": ["User Login"]\n  },\n  "State": "ENABLED",\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:MyFunction"\n    }\n  ]\n}\n'

In [24]:
import json

json.loads(text.strip())

{'Name': 'MyRule',
 'EventPattern': {'source': ['myapp'], 'detail-type': ['User Login']},
 'State': 'ENABLED',
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:MyFunction'}]}

In [ ]:
from tracemalloc import stop

messages = []

add_user_message(messages, "Generate three different sample AWS CLI commands. Each should be very short")

add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

# Stop-sequence
text = chat(messages, stop_sequences=["```"])

text

'\naws s3 ls\naws ec2 describe-instances\naws lambda list-functions\n'